<h1><b><center>Affiliate Project</center></b></h1>

<h3><b><center>Project Brief</center></b></h3>

Identify and retrieve the Player IDs of players who registered via an affiliate link. Store the players' Player IDs, Mobile Numbers, Affiliate IDs. and Country of origin in a CSV. Send CSV to an SFTP. 

<h3><b><center>Code & Logic</center></b></h3>

**1. Import Packages**

In [ ]:
# Package used to connect to MySQL Databases
import mysql.connector
import pymysql
import paramiko

#Connect To SFTP
import pysftp

# Folder Creation
import os
from pathlib import Path
from dotenv import load_dotenv

# Data Manipulation Packages
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Package To Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

**2. Get Credentials From .env File**

In [ ]:
ROOT_DIR: Path = Path().resolve().parent
load_dotenv(os.path.join(ROOT_DIR, ".env"))
Root = os.path.normpath(os.getcwd() + os.sep + os.pardir)

# Local DB Credentials
host_       = os.getenv('HOST')
database_   = os.getenv('DATABASE')
user_       = os.getenv('NAME')
password_   = os.getenv('PASSWORD')
port_       = os.getenv('PORT')

# SFTP Credentials
host        = os.getenv('SFT_HOST')
username    = os.getenv('SFT_NAME')
password    = os.getenv('SFT_PASSWORD')
port        = os.getenv('SFT_PORT')
sft_path    = os.getenv('SFT_PATH')

**3. Create SQL Query To Pull Data From Database**

In [ ]:
Query = """SELECT  
                 a.created_date
                ,a.affiliate_id
                ,a.player_id
                ,b.mobile_number
                ,b.player_country
        FROM (
                SELECT 
                         player_id
                        ,affiliate_id
                        ,created_date
                FROM bi_schema.affiliates
                WHERE created_date = CURDATE() - INTERVAL 1 DAY
                AND transaction_type = 'REGISTRATION'
                AND affiliate_id IS NOT NULL
        ) a
        LEFT JOIN bi_schema.player_data b
        ON a.player_id = b.player_id;
"""

**4. Import Data as Dataframe From MySQL Database**

In [ ]:
# Code To Connect MySQL
connection = mysql.connector.connect(host=host_
                                      ,database=database_
                                      ,user=user_
                                      ,password=password_
                                      ,port=port_)

# Connect to MySQL database
try:
    with connection.cursor() as cursor:
        df = pd.read_sql(Query,connection)
finally:
    connection.close()

In [ ]:
df.head()

**5. Create Function That Exports CSV(s) and Sends To SFTP**

In [ ]:
def export_files(df, column, Name):

    # SSH setup
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.load_system_host_keys()
    ssh.connect(hostname=host, port=port, username=username, password=password)
    print("SSH connection successful")

    sftp = ssh.open_sftp()
    print("SFTP connection successful")

    df[column] = pd.to_datetime(df[column]).dt.date

    def process_group(group):
        d = group.name  # group date

        year = d.year
        month_folder = f"{d.month:02d}_{d.strftime('%B')}"
        folder_path = os.path.join(str(year), month_folder)
        os.makedirs(folder_path, exist_ok=True)

        filename = f"{Name}_{d}.csv"
        filepath = os.path.join(folder_path, filename)

        group.to_csv(filepath, index=False)
        sftp.put(filepath, sft_path + filename)

    # Apply Logic
    df.groupby(column).apply(process_group)

    print("Listing directory contents after upload:")
    print(sftp.listdir())

    sftp.close()
    ssh.close()

**6. Call Function**

In [ ]:
export_files(df, Column='created_date', Name='affiliates_data')